In [1]:
!hostname

n103.clstr


In [2]:
import pandas as pd 
import requests
import io
import numpy as np
import matplotlib.pyplot as plt
import glob
import xarray as xr
import geopandas as gpd
import matplotlib.colors as mcolors
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [3]:
winter_months = [11, 12, 1, 2, 3]

RICK THOMAN
-

In [4]:
#Rick thoman data 
#https://docs.google.com/spreadsheets/d/1MUeq6N2F8OGBeYOa226iCbaVQBOxj01ibf5IwYqt5nI/edit?gid=0#gid=0
url = "https://docs.google.com/spreadsheets/d/1MUeq6N2F8OGBeYOa226iCbaVQBOxj01ibf5IwYqt5nI/export?format=csv&id=1MUeq6N2F8OGBeYOa226iCbaVQBOxj01ibf5IwYqt5nI&gid=0"
rick_df = pd.read_csv(url)

rick_df['Year'] = pd.to_numeric(rick_df['Year'], errors='coerce')
rick_df['Month'] = pd.to_numeric(rick_df['Month'], errors='coerce')
rick_df['Rain Precip Amount (mm)'] = pd.to_numeric(rick_df['Rain Precip Amount (mm)'],errors='coerce')

rick_season_df = rick_df.loc[rick_df['Month'].isin([11, 12, 1, 2, 3])].copy()
season_start = rick_season_df['Year'].where(~rick_season_df['Month'].isin([1, 2, 3]),rick_season_df['Year'] - 1)
season_end = season_start + 1

rick_season_df['season'] = (season_start.astype(int).astype(str)+ '-'+ season_end.astype(int).astype(str))
rick_season_df['season_sum_rain_mm'] = (rick_season_df.groupby('season')['Rain Precip Amount (mm)'].transform('sum'))

rick_monthly = rick_season_df.groupby('Month')['Rain Precip Amount (mm)'].sum(numeric_only=True)
rick_monthly=rick_monthly.reindex(winter_months).values #correct order 

In [5]:
rick_dec_2021=rick_season_df[rick_season_df['Year']==2021]
rick_dec_2021
rick_dec_2021_sum= rick_dec_2021['season_sum_rain_mm'].sum()
rick_dec_2021_sum

38.1

In [6]:
rick_nov_2010=rick_season_df[rick_season_df['Year']==2010]
rick_nov_2010
rick_nov_2010_sum= rick_nov_2010['season_sum_rain_mm'].sum()
rick_nov_2010_sum

24.13

Alaska Shapefiles 
-

In [7]:
shapefile_path = "/center1/DYNDOWN/phutton5/ROS/boundaries/Alaska_Borough_and_Census_Area_Boundaries.shp"
borough_boundaries = gpd.read_file(shapefile_path)
borough_boundaries = borough_boundaries.set_crs(epsg=3338)
borough_boundaries = borough_boundaries.to_crs(epsg=4326)
FNSB_boundary = borough_boundaries[borough_boundaries['CommunityN'] == 'Fairbanks North Star Borough']
FNSB_geom = FNSB_boundary.geometry.iloc[0] 
FNSB_coords = []
FNSB_coords.extend(list(FNSB_geom.exterior.coords))
FNSB_coords = np.array(FNSB_coords)  
FNSB_coords = pd.DataFrame({
    "lon": FNSB_coords[:, 0],
    "lat": FNSB_coords[:, 1]})

Fairbanks_lat=(64.84)
Fairbanks_lon=(-147.72)

ASOS 
-

In [8]:
website='https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?network=AK_ASOS&station=PAWI&data=p01m&data=wxcodes&data=metar&year1=1950&month1=1&day1=1&year2=2023&month2=3&day2=31&tz=Etc%2FUTC&format=onlycomma&latlon=yes&elev=yes&missing=M&trace=T&direct=no&report_type=3&report_type=4'
#'https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?network=AK_ASOS&station=PAFA&data=tmpc&data=p01m&data=wxcodes&data=snowdepth&year1=1949&month1=10&day1=1&year2=2022&month2=12&day2=23&tz=Etc%2FUTC&format=onlycomma&latlon=yes&elev=no&missing=M&trace=T&direct=no&report_type=3&report_type=4'
#https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?network=AK_ASOS&station=PAFA&data=tmpc&data=p01m&data=wxcodes&data=snowdepth&year1=1950&month1=1&day1=1&year2=2023&month2=12&day2=9&tz=America%2FAnchorage&format=onlycomma&latlon=yes&elev=no&missing=M&trace=T&direct=no&report_type=3&report_type=4'
response = requests.get(website)
if response.status_code == 200:
    data = io.StringIO(response.text)
    df = pd.read_csv(data, comment="#")  
    print(df.head())
else:
    print("Error:", response.status_code)

  station             valid       lon     lat  elevation p01m wxcodes  \
0    PAWI  1957-08-02 18:00 -159.9948  70.638       27.0    M       M   
1    PAWI  1957-08-03 00:00 -159.9948  70.638       27.0    M       M   
2    PAWI  1957-08-03 06:00 -159.9948  70.638       27.0    M       M   
3    PAWI  1957-08-03 12:00 -159.9948  70.638       27.0    M       M   
4    PAWI  1957-08-03 18:00 -159.9948  70.638       27.0    M       M   

                                               metar  
0  METAR PAWI 021800Z 25006KT 1/2SM OVC/// 06/05 ...  
1  METAR PAWI 030000Z 29006KT 10SM OVC/// 11/07 A...  
2  METAR PAWI 030600Z 32005KT 10SM BKN/// 11/09 A...  
3  METAR PAWI 031200Z 18007KT 10SM OVC/// 06/06 A...  
4  METAR PAWI 031800Z 02009KT 15SM BKN/// BKN/// ...  


In [9]:
df['valid'] = pd.to_datetime(df['valid'])
df['month'] = df['valid'].dt.month
df['date'] = df['valid'].dt.date
df['time'] = df['valid'].dt.time
#df['tmpc']
#df = df.drop(columns=['column_name'])
winter_df = df[df['month'].isin([11, 12, 1, 2, 3])] #filter to only keep the ROS months 

winter_df['date'] = pd.to_datetime(winter_df['date'])
year = winter_df['date'].dt.year
month = winter_df['date'].dt.month
season_start = year.where(~month.isin([1, 2, 3]), year - 1)
season_end = season_start + 1
winter_df['season'] = season_start.astype(str) + '-' + season_end.astype(str)

#filter to only when RA is present 
mask = winter_df['wxcodes'].str.contains('RA', na=False)
rain_and_mixed_df = winter_df[mask]
rain_and_mixed_df['p01m'] = rain_and_mixed_df['p01m'].replace('M', np.nan) 
#rain_and_mixed_df['p01m'] = rain_and_mixed_df['p01m'].replace('T', np.nan)
rain_and_mixed_df['p01m'] = rain_and_mixed_df['p01m'].replace('T', 0.1)
rain_and_mixed_df['p01m'] = pd.to_numeric(rain_and_mixed_df['p01m'], errors='coerce')

/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2253879/4063370908.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  winter_df['date'] = pd.to_datetime(winter_df['date'])
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2253879/4063370908.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  winter_df['season'] = season_start.astype(str) + '-' + season_end.astype(str)
/center1/DYNDOWN/phutton5/VSCODE_dump/ipykernel_2253879/4063370908.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a

In [10]:
filtered_rain_and_mixed_df = rain_and_mixed_df[(rain_and_mixed_df['p01m'].notna()) & (rain_and_mixed_df['p01m'] != 0)]
filtered_rain_and_mixed_df=(filtered_rain_and_mixed_df.groupby(['season', 'month'])['p01m'].sum().reset_index())

p90 = filtered_rain_and_mixed_df['p01m'].quantile(0.90)
p95 = filtered_rain_and_mixed_df['p01m'].quantile(0.95)
p75 = filtered_rain_and_mixed_df['p01m'].quantile(0.75)
p80 = filtered_rain_and_mixed_df['p01m'].quantile(0.80)
p50 = filtered_rain_and_mixed_df['p01m'].quantile(0.5)
print(p50, p75, p80, p90, p95)
#filtered_greatherthan_01_rain_and_mixed_df=filtered_rain_and_mixed_df[filtered_rain_and_mixed_df['p01m'] > 2.54]

filtered_90percentile_rain_and_mixed_df=filtered_rain_and_mixed_df[filtered_rain_and_mixed_df['p01m'] > p90]
filtered_90percentile_rain_and_mixed_df

0.6000000000000001 1.7000000000000002 2.0400000000000014 3.1399999999999992 4.805


,season,month,p01m
22,2009-2010,12,10.40
26,2010-2011,11,11.59
29,2012-2013,1,4.70
45,2018-2019,11,4.85
46,2018-2019,12,3.50


To Compare PAFA, DYN and RAW
-


In [11]:
pafa_lat=70.6380
pafa_lon=-159.9948

#-159.9948	70.638	
#actually wainright not APFA

In [12]:
def getXY(lat, lon, dataarray):
    abslat = np.abs(dataarray.XLAT-lat)
    abslon = np.abs(dataarray.XLONG-lon)
    d = abslon**2 + abslat**2
    flat_index = np.argmin(d.values)
    yloc, xloc = np.unravel_index(flat_index, d.shape)
    return xloc, yloc

In [14]:
def getXY(lat, lon, dataarray):
    abslat = np.abs(dataarray.XLAT-lat)
    abslon = np.abs(dataarray.XLONG-lon)
    d = abslon**2 + abslat**2
    flat_index = np.argmin(d.values)
    yloc, xloc = np.unravel_index(flat_index, d.shape)
    return xloc, yloc

def getXY_latlon(lat, lon, dataarray):
    abslat = np.abs(dataarray.latitude - lat)
    abslon = np.abs(dataarray.longitude - lon)
    d = abslon**2 + abslat**2
    flat_index = np.argmin(d.values)
    yloc, xloc = np.unravel_index(flat_index, d.shape)
    return xloc, yloc

In [19]:
x_idx, y_idx = getXY(pafa_lat, pafa_lon, regridded_era5)
nearest_lat = regridded_era5.XLAT[y_idx, x_idx]
nearest_lon = regridded_era5.XLONG[y_idx, x_idx]


In [20]:
regridded_era5_path='/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/ERA5_31kmto4km_nearest_regridded.nc'
regridded_era5=xr.open_dataset(regridded_era5_path)

#ERA5  4km
era5_4km='/center1/DYNDOWN/phutton5/ROS/All_of_AK/All_of_AK_netcdf_files/ROS_Monthly_*.nc'
era5_4km = xr.open_mfdataset(era5_4km,combine="by_coords", parallel=True)


In [21]:
cell = era5_4km.isel(
    south_north=y_idx,
    west_east=x_idx
)

In [22]:
era5_4km_rain_sum_at_site = era5_4km['rain_ros_sum'].isel(south_north=y_idx,west_east=x_idx)
seasonal_rain_sum_at_site_era5_4km=era5_4km_rain_sum_at_site.sum(dim='month')
monthly_rain_sum_at_site_era5_4km=era5_4km_rain_sum_at_site.sum(dim='season')
#monthly_rain_MEAN_at_site_era5_4km=era5_4km['rain_ros_sum'].isel(south_north=y_idx,west_east=x_idx).groupby('month','season').mean()


era5_regridded_31km_rain_sum_at_site = regridded_era5['rain_ros_sum'].isel(south_north=y_idx,west_east=x_idx)
seasonal_rain_sum_at_site_era5_regridded_31km=era5_regridded_31km_rain_sum_at_site.sum(dim='month')
monthly_rain_sum_at_site_era5_regridded_31km=era5_regridded_31km_rain_sum_at_site.sum(dim='season')

#monthly_rain_MEAN_at_site_era5_regridded_31km=regridded_era5['rain_ros_sum'].isel(south_north=y_idx,west_east=x_idx).groupby('month').mean()

seasons=era5_4km['season']

In [23]:
# 4km ERA5 as DataFrame
era5_4km_df = era5_4km_rain_sum_at_site.to_dataframe().reset_index()
era5_31km_df = era5_regridded_31km_rain_sum_at_site.to_dataframe().reset_index() 
era5_4km_monthly = era5_4km_df.groupby(['season', 'month'])['rain_ros_sum'].sum().reset_index()
era5_31km_monthly = era5_31km_df.groupby(['season', 'month'])['rain_ros_sum'].sum().reset_index()

top_rain_df = filtered_90percentile_rain_and_mixed_df.copy()

top_rain_df = top_rain_df.merge(
    era5_4km_monthly,
    on=['season', 'month'],
    how='left'
).rename(columns={'rain_ros_sum':'ERA5_4km_mm'})

top_rain_df = top_rain_df.merge(
    era5_31km_monthly,
    on=['season', 'month'],
    how='left'
).rename(columns={'rain_ros_sum':'ERA5_31km_mm'})

top_rain_df

,season,month,p01m,ERA5_4km_mm,ERA5_31km_mm
0,2009-2010,12,10.40,0.257233,0.00000
1,2010-2011,11,11.59,1.402945,0.76506
2,2012-2013,1,4.70,0.000000,0.00000
3,2018-2019,11,4.85,0.000000,0.00000
4,2018-2019,12,3.50,0.000000,0.00000
